# NT-v2 On TSS Windows, Colab A100

This notebook runs the CDS-vs-TSS ablation for the paper:

- Fetch/cache 196,608 bp TSS-centered windows.
- Extract NT-v2 multi-pool features on those TSS windows.
- Build `dataset_tss_nt_v2_*` parquets.
- Run family5 logistic probes and Ridge-to-GenePT probes.
- Package outputs to download back into the repo.

Before running: `Runtime -> Change runtime type -> A100 GPU`.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Clone The Branch

This notebook uses the active branch that contains the TSS scripts.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Austin-Senna/dna_to_text.git'
BRANCH = 'codex/dna-encoder-expansion'
REPO_DIR = Path('/content/dna_to_text')

if REPO_DIR.exists():
    %cd /content/dna_to_text
    !git fetch origin
    !git checkout $BRANCH
    !git pull
else:
    !git clone -b $BRANCH $REPO_URL /content/dna_to_text
    %cd /content/dna_to_text

!git rev-parse --abbrev-ref HEAD
!git rev-parse --short HEAD

## Install Dependencies

Colab already has CUDA PyTorch. This installs the repo package and its Python dependencies.

In [ ]:
!pip install -q -U pip
!pip install -q -e .

## Optional: Persist Hugging Face Cache In Drive

Use this if you expect to rerun the notebook. It avoids redownloading model files. You can skip it.

In [ ]:
USE_DRIVE_CACHE = False

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
    os.environ['TRANSFORMERS_CACHE'] = '/content/drive/MyDrive/hf_cache'
    print('HF_HOME:', os.environ['HF_HOME'])
else:
    print('Using default Colab Hugging Face cache')

## Fetch TSS Windows

This uses the Enformer window code only to fetch/cache TSS windows and build the matched TSS 4-mer dataset. `--skip-model` means it does not load or run Enformer.

In [ ]:
!python scripts/run_enformer_features.py \
  --template-dataset data/dataset_nt_v2_meanD.parquet \
  --skip-model

In [ ]:
!find data/enformer_windows -maxdepth 1 -name '*.fa' | wc -l
!ls -lh data/dataset_enformer_tss_4mer.parquet

## Optional Pilot Extraction

Run this first if you want a quick sanity check. It caches the first 25 genes. The full run below will skip those cached genes.

In [ ]:
RUN_PILOT = False

if RUN_PILOT:
    !python scripts/run_tss_multi_pool_extract.py \
      --encoder nt_v2 \
      --template-dataset data/dataset_nt_v2_meanD.parquet \
      --device cuda \
      --max-genes 25
else:
    print('Skipping pilot. Set RUN_PILOT = True to run it.')

## Full NT-v2 TSS Extraction

This is the long step. It writes one cached `.npz` per gene under `data/tss_chunk_reductions_nt_v2/` and is resumable.

In [ ]:
!python scripts/run_tss_multi_pool_extract.py \
  --encoder nt_v2 \
  --template-dataset data/dataset_nt_v2_meanD.parquet \
  --device cuda

In [ ]:
!find data/tss_chunk_reductions_nt_v2 -maxdepth 1 -name '*.npz' | wc -l
!du -sh data/tss_chunk_reductions_nt_v2

## Build TSS Pooling Datasets

In [ ]:
!python scripts/build_tss_pooling_datasets.py \
  --encoder nt_v2 \
  --template-dataset data/dataset_nt_v2_meanD.parquet \
  --variants meanmean meanD meanG maxmean clsmean

!ls -lh data/dataset_tss_nt_v2*.parquet

## Run Main Family5 And Ridge Probes

This runs the three most important pooling variants first.

In [ ]:
%%bash
set -euo pipefail

for v in meanmean meanD meanG; do
  python scripts/train_logistic_probe.py --dataset "tss_nt_v2_${v}" --task family5
  python scripts/train_probe.py \
    --dataset "data/dataset_tss_nt_v2_${v}.parquet" \
    --probe-out "data/probe_tss_nt_v2_${v}.npz"
done

## Optional Full Pooling Sweep

In [ ]:
RUN_OPTIONAL_POOLING = False

if RUN_OPTIONAL_POOLING:
    !python scripts/train_logistic_probe.py --dataset tss_nt_v2_maxmean --task family5
    !python scripts/train_probe.py --dataset data/dataset_tss_nt_v2_maxmean.parquet --probe-out data/probe_tss_nt_v2_maxmean.npz
    !python scripts/train_logistic_probe.py --dataset tss_nt_v2_clsmean --task family5
    !python scripts/train_probe.py --dataset data/dataset_tss_nt_v2_clsmean.parquet --probe-out data/probe_tss_nt_v2_clsmean.npz
else:
    print('Skipping optional maxmean/clsmean probes.')

## Rebuild Tables And Inspect TSS Rows

In [ ]:
!python scripts/build_family5_table.py
!python scripts/build_regression_table.py

!sed -n '/TSS Self-Supervised Encoder Ablation/,$p' data/family5_table.md | head -n 24
!sed -n '/TSS Self-Supervised Encoder Ablation/,$p' data/regression_table.md | head -n 28

## Package Outputs For Download

The minimal bundle is enough to bring the results back to the local repo. The full cache bundle is larger but lets you resume/materialize without rerunning extraction.

In [ ]:
!tar -czf ntv2_tss_minimal_outputs.tar.gz \
  data/dataset_tss_nt_v2*.parquet \
  data/probe_tss_nt_v2*.npz \
  data/confusion_5way_tss_nt_v2*.json \
  data/metrics.json \
  data/family5_table.md \
  data/regression_table.md

!ls -lh ntv2_tss_minimal_outputs.tar.gz

In [ ]:
from google.colab import files
files.download('ntv2_tss_minimal_outputs.tar.gz')